# Page One Builder: 10 Keywords on Google Maps (SEO + AEO)

Pick a restaurant, generate 20 local keywords with Gemini, pick the top 10, and get Google Business Profile changes, website copy, AEO FAQs, validated JSON-LD schema, a rank tracker and an HTML report.

**Keys rule:** add `GEMINI_API_KEY` in Colab's 🔑 Secrets panel (left sidebar). The key is never typed into this notebook. No key? Set `OFFLINE = True` to use the cached sample output.

In [ ]:
%pip install -q google-genai python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY') or ''
    print('Key loaded from Colab secrets:', bool(os.environ['GEMINI_API_KEY']))
except Exception as e:
    print('Not in Colab or no secret set; will use env/.env or offline mode.')

In [ ]:
# Write the tool and the cached sample output
import pathlib
pathlib.Path('sample_output').mkdir(exist_ok=True)
pathlib.Path('page_one.py').write_text('#!/usr/bin/env python3\n"""Page One Builder: 10 keywords on Google Maps (SEO + AEO).\n\nGenerates a 10-keyword local search plan for a restaurant: Google Business Profile\nchanges, website page copy, AEO FAQs, validated JSON-LD schema, a manual rank\ntracker, an HTML report, and a landing page with schema in <head>.\n\nKeys rule: the Gemini key is read ONLY from the environment (GEMINI_API_KEY),\na local .env file (gitignored), or Colab secrets. It is never written anywhere.\n\nUsage:\n  python page_one.py --restaurant "Sample Chettinad Kitchen"\n  python page_one.py --csv data/restaurants.csv --row 0\n  python page_one.py --offline          # cached sample output, no key / wifi needed\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport html\nimport json\nimport os\nimport re\nimport sys\nfrom datetime import date\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parent\nSAMPLE_JSON = ROOT / "sample_output" / "sample_chettinad_kitchen.plan.json"\n\nSAMPLE_RESTAURANT = {\n    "name": "Sample Chettinad Kitchen",\n    "address": "Richardson, TX 75080",\n    "cuisine": "South Indian (Chettinad)",\n    "website": "https://example.com/sample-chettinad-kitchen",\n    "menu_link": "https://example.com/sample-chettinad-kitchen/menu",\n    "phone": "",\n    "hours": "Mo-Su 11:00-23:00",\n    "features": "halal; lunch buffet; open till 11 PM; near UT Dallas",\n    "menu_items": "chicken biryani; chettinad chicken; dosa",\n}\n\nINTENT_GROUPS = ["dish", "cuisine_near_me", "occasion_feature"]\n\n# ----------------------------------------------------------------------------\n# Input\n# ----------------------------------------------------------------------------\n\ndef load_csv_row(path: str, row: int | None, name: str | None) -> dict:\n    with open(path, newline="", encoding="utf-8-sig") as f:\n        rows = [{(k or "").strip().lower(): (v or "").strip() for k, v in r.items()} for r in csv.DictReader(f)]\n    if not rows:\n        sys.exit(f"No rows in {path}")\n    if name:\n        for r in rows:\n            if r.get("name", "").lower() == name.lower():\n                return normalize(r)\n        sys.exit(f"Restaurant \'{name}\' not found in {path}")\n    idx = row or 0\n    if idx >= len(rows):\n        sys.exit(f"Row {idx} out of range (CSV has {len(rows)} rows, 0-indexed)")\n    return normalize(rows[idx])\n\n\ndef normalize(r: dict) -> dict:\n    out = dict(SAMPLE_RESTAURANT) if r.get("name") == SAMPLE_RESTAURANT["name"] else {k: "" for k in SAMPLE_RESTAURANT}\n    aliases = {"menu": "menu_link", "menu link": "menu_link", "telephone": "phone", "items": "menu_items"}\n    for k, v in r.items():\n        k = aliases.get(k, k.replace(" ", "_"))\n        if v:\n            out[k] = v\n    return out\n\n\ndef split_list(s: str) -> list[str]:\n    return [x.strip() for x in re.split(r"[;|]", s or "") if x.strip()]\n\n\ndef city_of(address: str) -> str:\n    m = re.search(r"([A-Za-z .]+),\\s*([A-Z]{2})", address or "")\n    return f"{m.group(1).strip().split(\',\')[-1].strip()}, {m.group(2)}" if m else address\n\n# ----------------------------------------------------------------------------\n# Gemini\n# ----------------------------------------------------------------------------\n\nPROMPT = """You are a local SEO + AEO (answer-engine optimization) strategist for restaurants.\nRestaurant data (only use these facts; do NOT invent awards, reviews, prices or claims):\n{restaurant}\n\nTask:\n1. Generate exactly 20 realistic local search keywords that people type into Google Maps\n   or ask AI assistants (ChatGPT, Gemini, Siri) about this kind of restaurant in this city.\n   Group each into intent_group: "dish", "cuisine_near_me", or "occasion_feature".\n   Give est_intent: "high" | "medium" | "low" (likelihood the searcher visits/orders soon),\n   intent_score 1-10, and a one-line why.\n2. Pick the top 10 (mix of all three groups, highest intent and best fit for the restaurant\'s real menu/features).\n   For each top keyword return:\n   - gbp: categories (primary first), description_addition (1-2 natural sentences, <=750 chars total description),\n     menu_items (items to add/rename on the GBP menu), photos (shot list), posts (2 Google Posts ideas),\n     review_request (an honest way to ask real customers for reviews; never incentivize or script reviews)\n   - website: title_tag (<=60 chars), meta_description (<=155 chars), h1, page_copy (300-500 words, natural,\n     mention menu items, prices as "[price]" placeholders if unknown, hours, location; no keyword stuffing)\n   - aeo_faqs: 3 Q&A pairs, short factual answers an AI assistant can quote.\n3. faqs: the 5 best overall FAQs for a FAQPage schema block.\nHonest methods only: no fake reviews, fake clicks, keyword stuffing or ranking manipulation.\n\nReturn JSON with this exact shape:\n{{"keywords":[{{"keyword":"","intent_group":"","est_intent":"","intent_score":0,"why":""}}],\n "top10":[{{"rank":1,"keyword":"","intent_group":"","gbp":{{"categories":[],"description_addition":"","menu_items":[],"photos":[],"posts":[],"review_request":""}},\n   "website":{{"title_tag":"","meta_description":"","h1":"","page_copy":""}},"aeo_faqs":[{{"q":"","a":""}}]}}],\n "gbp_description":"full suggested GBP description <=750 chars",\n "faqs":[{{"q":"","a":""}}]}}\n"""\n\n\ndef get_api_key() -> str | None:\n    try:\n        from dotenv import load_dotenv\n        load_dotenv(ROOT / ".env")\n    except ImportError:\n        pass\n    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")\n    if not key:\n        try:  # Colab secrets\n            from google.colab import userdata  # type: ignore\n            key = userdata.get("GEMINI_API_KEY")\n            if key:\n                os.environ["GEMINI_API_KEY"] = key\n        except Exception:\n            pass\n    return key\n\n\ndef call_gemini(restaurant: dict, model: str) -> dict:\n    from google import genai\n    from google.genai import types\n\n    client = genai.Client(api_key=get_api_key())\n    cfg = dict(response_mime_type="application/json", temperature=0.4, max_output_tokens=32000)\n    try:\n        cfg["thinking_config"] = types.ThinkingConfig(thinking_budget=0)\n        config = types.GenerateContentConfig(**cfg)\n    except Exception:\n        cfg.pop("thinking_config", None)\n        config = types.GenerateContentConfig(**cfg)\n    resp = client.models.generate_content(\n        model=model,\n        contents=PROMPT.format(restaurant=json.dumps(restaurant, indent=2)),\n        config=config,\n    )\n    text = (resp.text or "").strip()\n    text = re.sub(r"^```(?:json)?|```$", "", text).strip()\n    return json.loads(text)\n\n# ----------------------------------------------------------------------------\n# Schema (built deterministically from real data, then validated)\n# ----------------------------------------------------------------------------\n\ndef build_restaurant_schema(r: dict, plan: dict) -> dict:\n    addr = r.get("address", "")\n    m = re.match(r"^(?:(?P<street>.+?),\\s*)?(?P<city>[^,]+),\\s*(?P<state>[A-Z]{2})\\s*(?P<zip>\\d{5})?", addr)\n    postal = {"@type": "PostalAddress", "addressCountry": "US"}\n    if m:\n        if m.group("street"):\n            postal["streetAddress"] = m.group("street")\n        postal["addressLocality"] = m.group("city").strip()\n        postal["addressRegion"] = m.group("state")\n        if m.group("zip"):\n            postal["postalCode"] = m.group("zip")\n    else:\n        postal["streetAddress"] = addr\n    s = {\n        "@context": "https://schema.org",\n        "@type": "Restaurant",\n        "name": r["name"],\n        "servesCuisine": [c.strip() for c in re.split(r"[,(/)]", r.get("cuisine", "")) if c.strip()] or r.get("cuisine"),\n        "address": postal,\n        "telephone": r.get("phone") or "+1-000-000-0000 (replace with real number)",\n        "openingHours": [h.strip() for h in r.get("hours", "").split(";") if h.strip()] or ["Mo-Su 11:00-23:00"],\n        "menu": r.get("menu_link") or r.get("website"),\n        "acceptsReservations": "True",\n        "url": r.get("website"),\n        "priceRange": "$$",\n        "description": plan.get("gbp_description", "")[:750],\n        "hasMenu": {\n            "@type": "Menu",\n            "hasMenuSection": {\n                "@type": "MenuSection",\n                "name": "Signature dishes",\n                "hasMenuItem": [{"@type": "MenuItem", "name": i.title()} for i in split_list(r.get("menu_items", ""))],\n            },\n        },\n    }\n    feats = split_list(r.get("features", ""))\n    if feats:\n        s["amenityFeature"] = [{"@type": "LocationFeatureSpecification", "name": f, "value": True} for f in feats]\n    return s\n\n\ndef build_faq_schema(faqs: list[dict]) -> dict:\n    return {\n        "@context": "https://schema.org",\n        "@type": "FAQPage",\n        "mainEntity": [\n            {"@type": "Question", "name": f["q"], "acceptedAnswer": {"@type": "Answer", "text": f["a"]}}\n            for f in faqs[:5]\n        ],\n    }\n\n\ndef validate_schema(restaurant_ld: dict, faq_ld: dict) -> list[str]:\n    """Round-trip both blocks through the JSON parser and check required fields."""\n    problems = []\n    for label, block in (("Restaurant", restaurant_ld), ("FAQPage", faq_ld)):\n        try:\n            json.loads(json.dumps(block))\n        except Exception as e:  # pragma: no cover\n            problems.append(f"{label}: invalid JSON ({e})")\n        if block.get("@context") != "https://schema.org":\n            problems.append(f"{label}: missing @context")\n    for k in ["name", "servesCuisine", "address", "telephone", "openingHours", "menu", "acceptsReservations"]:\n        if not restaurant_ld.get(k):\n            problems.append(f"Restaurant: missing {k}")\n    if "(replace" in str(restaurant_ld.get("telephone")):\n        problems.append("Restaurant: telephone is a placeholder - add the real number before publishing (warning)")\n    if len(faq_ld.get("mainEntity", [])) != 5:\n        problems.append(f"FAQPage: expected 5 questions, got {len(faq_ld.get(\'mainEntity\', []))}")\n    return problems\n\n# ----------------------------------------------------------------------------\n# Honesty lint\n# ----------------------------------------------------------------------------\n\nBANNED = ["fake review", "buy review", "paid review", "click farm", "best in the world", "#1 rated", "guaranteed ranking"]\n\n\ndef lint_plan(plan: dict) -> list[dict]:\n    out = []\n    for t in plan.get("top10", []):\n        copy = t.get("website", {}).get("page_copy", "")\n        words = re.findall(r"\\w+", copy.lower())\n        n = len(words) or 1\n        kw = t["keyword"].lower()\n        hits = copy.lower().count(kw)\n        density = round(100 * hits * len(kw.split()) / n, 1)\n        flags = []\n        if not 300 <= n <= 500:\n            flags.append(f"page copy {n} words (target 300-500)")\n        if density > 3:\n            flags.append(f"keyword density {density}% (>3% = stuffing risk)")\n        for b in BANNED:\n            if b in json.dumps(t).lower():\n                flags.append(f"banned phrase: {b}")\n        tt = t.get("website", {}).get("title_tag", "")\n        md = t.get("website", {}).get("meta_description", "")\n        if len(tt) > 60:\n            flags.append(f"title tag {len(tt)} chars (>60)")\n        if len(md) > 155:\n            flags.append(f"meta description {len(md)} chars (>155)")\n        out.append({"keyword": t["keyword"], "words": n, "density_pct": density, "flags": flags})\n    return out\n\n# ----------------------------------------------------------------------------\n# Outputs\n# ----------------------------------------------------------------------------\n\ndef slugify(s: str) -> str:\n    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")\n\n\ndef write_csvs(outdir: Path, plan: dict, r: dict):\n    with open(outdir / "keywords_20.csv", "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow(["keyword", "intent_group", "est_intent", "intent_score", "in_top10", "why"])\n        top = {t["keyword"].lower() for t in plan["top10"]}\n        for k in plan["keywords"]:\n            w.writerow([k["keyword"], k["intent_group"], k["est_intent"], k["intent_score"], k["keyword"].lower() in top, k.get("why", "")])\n    with open(outdir / "rank_tracker.csv", "w", newline="") as f:\n        w = csv.writer(f)\n        w.writerow([\n            "rank", "keyword", "intent_group", "search_location",\n            "before_date", "before_maps_position(1-20 or \'not found\')", "before_in_maps_top3(Y/N)",\n            "before_ai_assistant_mentions(Y/N, which: ChatGPT/Gemini/Perplexity)", "before_ai_answer_notes",\n            "changes_made", "after_date", "after_maps_position", "after_in_maps_top3(Y/N)", "after_ai_assistant_mentions(Y/N)", "notes",\n        ])\n        for t in plan["top10"]:\n            w.writerow([t["rank"], t["keyword"], t["intent_group"], f"{city_of(r.get(\'address\',\'\'))} (incognito, location set)",\n                        "", "", "", "", "", "", "", "", "", "", ""])\n\n\ndef esc(s) -> str:\n    return html.escape(str(s or ""))\n\n\ndef paragraphs(text: str) -> str:\n    return "".join(f"<p>{esc(p.strip())}</p>" for p in re.split(r"\\n\\s*\\n|\\n", text or "") if p.strip())\n\n\nCSS = """\n:root{--ink:#1d1d1f;--muted:#6e6e73;--line:#e5e5ea;--acc:#c2410c;--acc2:#0f766e;--bg:#fafaf7}\n*{box-sizing:border-box}body{margin:0;font:16px/1.55 -apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;color:var(--ink);background:var(--bg)}\n.wrap{max-width:1060px;margin:0 auto;padding:32px 20px 80px}\nheader.hero{background:linear-gradient(135deg,#7c2d12,#c2410c 60%,#ea580c);color:#fff;padding:40px 20px}\nheader.hero h1{margin:0 0 6px;font-size:34px}header.hero p{margin:0;opacity:.9}\n.badges span{display:inline-block;background:rgba(255,255,255,.18);border-radius:999px;padding:4px 12px;margin:10px 6px 0 0;font-size:13px}\nh2{margin:40px 0 12px;font-size:24px}h3{margin:0 0 8px}\n.card{background:#fff;border:1px solid var(--line);border-radius:14px;padding:20px;margin:14px 0;box-shadow:0 1px 2px rgba(0,0,0,.03)}\n.grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:14px}\n.stat{font-size:30px;font-weight:700;color:var(--acc)}.muted{color:var(--muted)}\ntable{width:100%;border-collapse:collapse;font-size:14px}th,td{text-align:left;padding:8px 10px;border-bottom:1px solid var(--line);vertical-align:top}\nth{background:#f4f4f0;font-weight:600}\n.pill{display:inline-block;padding:2px 10px;border-radius:999px;font-size:12px;font-weight:600}\n.g-dish{background:#fff1e6;color:#9a3412}.g-cuisine_near_me{background:#e6f4f1;color:#115e59}.g-occasion_feature{background:#eef0ff;color:#3730a3}\n.i-high{color:#15803d;font-weight:700}.i-medium{color:#a16207}.i-low{color:#6b7280}\ndetails{background:#fff;border:1px solid var(--line);border-radius:14px;margin:12px 0;padding:0 18px}\nsummary{cursor:pointer;padding:16px 0;font-weight:600;font-size:17px;list-style:none}summary::-webkit-details-marker{display:none}\nsummary .n{display:inline-block;width:30px;height:30px;line-height:30px;text-align:center;border-radius:50%;background:var(--acc);color:#fff;margin-right:10px;font-size:14px}\n.cols{display:grid;grid-template-columns:1fr 1fr;gap:18px;padding-bottom:18px}@media(max-width:800px){.cols{grid-template-columns:1fr}}\n.box{background:#fafaf7;border:1px solid var(--line);border-radius:10px;padding:14px}.box h4{margin:0 0 8px;color:var(--acc2);font-size:14px;text-transform:uppercase;letter-spacing:.04em}\nul{margin:6px 0;padding-left:20px}pre{background:#0f172a;color:#e2e8f0;padding:16px;border-radius:10px;overflow:auto;font-size:12.5px;line-height:1.45}\n.ok{color:#15803d;font-weight:700}.warn{color:#b45309}.faq dt{font-weight:600;margin-top:10px}.faq dd{margin:4px 0 0 0;color:#333}\n.serp{font-family:arial,sans-serif;background:#fff;border:1px solid var(--line);border-radius:10px;padding:12px 14px}\n.serp .t{color:#1a0dab;font-size:18px}.serp .u{color:#006621;font-size:13px}.serp .d{color:#4d5156;font-size:14px}\n.banner{background:#fef3c7;border:1px solid #fcd34d;border-radius:10px;padding:10px 14px;margin:14px 0;font-size:14px}\n.honest{background:#ecfdf5;border:1px solid #a7f3d0;border-radius:14px;padding:18px}\n"""\n\n\ndef render_report(r: dict, plan: dict, rest_ld: dict, faq_ld: dict, problems: list[str], lint: list[dict], meta: dict) -> str:\n    groups = {g: [k for k in plan["keywords"] if k["intent_group"] == g] for g in INTENT_GROUPS}\n    top = {t["keyword"].lower() for t in plan["top10"]}\n    kw_rows = "".join(\n        f"<tr><td>{\'⭐ \' if k[\'keyword\'].lower() in top else \'\'}{esc(k[\'keyword\'])}</td>"\n        f"<td><span class=\'pill g-{esc(k[\'intent_group\'])}\'>{esc(k[\'intent_group\'].replace(\'_\',\' \'))}</span></td>"\n        f"<td class=\'i-{esc(k[\'est_intent\'])}\'>{esc(k[\'est_intent\'])}</td><td>{esc(k[\'intent_score\'])}/10</td><td class=\'muted\'>{esc(k.get(\'why\'))}</td></tr>"\n        for k in sorted(plan["keywords"], key=lambda k: -int(k.get("intent_score", 0)))\n    )\n    lint_by = {l["keyword"]: l for l in lint}\n    items = []\n    for t in plan["top10"]:\n        g, w = t["gbp"], t["website"]\n        li = lint_by.get(t["keyword"], {})\n        lint_html = ("<span class=\'ok\'>✓ passes honesty lint</span>" if not li.get("flags") else\n                     "<span class=\'warn\'>⚠ " + esc("; ".join(li["flags"])) + "</span>")\n        faqs = "".join(f"<dt>Q: {esc(f[\'q\'])}</dt><dd>A: {esc(f[\'a\'])}</dd>" for f in t.get("aeo_faqs", []))\n        items.append(f"""\n<details {\'open\' if t[\'rank\']==1 else \'\'}><summary><span class=\'n\'>{t[\'rank\']}</span>{esc(t[\'keyword\'])}\n <span class=\'pill g-{esc(t[\'intent_group\'])}\' style=\'margin-left:8px\'>{esc(t[\'intent_group\'].replace(\'_\',\' \'))}</span></summary>\n<div class=\'cols\'>\n <div class=\'box\'><h4>Google Business Profile</h4>\n  <b>Categories:</b> {esc(\', \'.join(g.get(\'categories\', [])))}<br>\n  <b>Description addition:</b> {esc(g.get(\'description_addition\'))}\n  <br><b>Menu items:</b><ul>{\'\'.join(f\'<li>{esc(x)}</li>\' for x in g.get(\'menu_items\', []))}</ul>\n  <b>Photos to add:</b><ul>{\'\'.join(f\'<li>{esc(x)}</li>\' for x in g.get(\'photos\', []))}</ul>\n  <b>Google Posts:</b><ul>{\'\'.join(f\'<li>{esc(x)}</li>\' for x in g.get(\'posts\', []))}</ul>\n  <b>Honest review request:</b> {esc(g.get(\'review_request\'))}\n </div>\n <div class=\'box\'><h4>Website page</h4>\n  <div class=\'serp\'><div class=\'u\'>{esc(r.get(\'website\'))}/{slugify(t[\'keyword\'])}</div><div class=\'t\'>{esc(w.get(\'title_tag\'))}</div><div class=\'d\'>{esc(w.get(\'meta_description\'))}</div></div>\n  <p><b>H1:</b> {esc(w.get(\'h1\'))}</p>\n  <p class=\'muted\'>{li.get(\'words\',\'?\')} words · keyword density {li.get(\'density_pct\',\'?\')}% · {lint_html}</p>\n  <details style=\'border:none;padding:0\'><summary style=\'font-size:14px;padding:6px 0\'>▸ Show full page copy</summary>{paragraphs(w.get(\'page_copy\'))}</details>\n  <h4 style=\'margin-top:12px\'>AEO: answers AI assistants can quote</h4><dl class=\'faq\'>{faqs}</dl>\n </div>\n</div></details>""")\n    prob_html = ("<p class=\'ok\'>✓ Both JSON-LD blocks parse as valid JSON and include all required fields.</p>"\n                 if not [p for p in problems if \'warning\' not in p] else "")\n    prob_html += "".join(f"<p class=\'warn\'>⚠ {esc(p)}</p>" for p in problems)\n    sample_banner = (f"<div class=\'banner\'><b>SAMPLE OUTPUT (offline/demo mode).</b> {esc(meta.get(\'source_note\',\'\'))}</div>"\n                     if meta.get("mode") == "offline" else "")\n    faq_all = "".join(f"<dt>{esc(f[\'q\'])}</dt><dd>{esc(f[\'a\'])}</dd>" for f in plan["faqs"][:5])\n    return f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>Page One Plan: {esc(r[\'name\'])}</title><style>{CSS}</style></head><body>\n<header class="hero"><div class="wrap" style="padding:0 20px">\n<h1>Page One Plan: {esc(r[\'name\'])}</h1>\n<p>10 keywords to win on Google Maps and in AI-assistant answers (SEO + AEO) · {esc(r.get(\'cuisine\'))} · {esc(r.get(\'address\'))}</p>\n<div class="badges"><span>Generated {esc(meta.get(\'date\'))}</span><span>Engine: {esc(meta.get(\'engine\'))}</span><span>Honest methods only</span></div>\n</div></header>\n<div class="wrap">{sample_banner}\n<div class="grid">\n <div class="card"><div class="stat">{len(plan[\'keywords\'])}</div>keywords researched</div>\n <div class="card"><div class="stat">10</div>picked to target</div>\n <div class="card"><div class="stat">{sum(len(t.get(\'aeo_faqs\',[])) for t in plan[\'top10\'])+5}</div>AEO answers written</div>\n <div class="card"><div class="stat">2</div>JSON-LD blocks {\'validated ✓\' if not [p for p in problems if \'warning\' not in p] else \'(check)\'}</div>\n</div>\n<div class="card"><h3>Restaurant</h3><b>{esc(r[\'name\'])}</b> · {esc(r.get(\'cuisine\'))}<br>{esc(r.get(\'address\'))}<br>\nSignature: {esc(\', \'.join(split_list(r.get(\'menu_items\',\'\'))))}<br>Features: {esc(\', \'.join(split_list(r.get(\'features\',\'\'))))}<br>\nHours: {esc(r.get(\'hours\'))} · Menu: {esc(r.get(\'menu_link\'))}</div>\n\n<h2>1. Keyword engine: {len(plan[\'keywords\'])} local searches, grouped by intent</h2>\n<p class="muted">Dish searches: {len(groups[\'dish\'])} · Cuisine + near me: {len(groups[\'cuisine_near_me\'])} · Occasion / feature: {len(groups[\'occasion_feature\'])}. ⭐ = picked for the top 10.</p>\n<div class="card" style="padding:0;overflow:auto"><table><tr><th>Keyword</th><th>Intent group</th><th>Est. intent</th><th>Score</th><th>Why</th></tr>{kw_rows}</table></div>\n\n<h2>2. The 10-keyword plan</h2>\n<p class="muted">For each keyword: what to change on the Google Business Profile, the website page to publish, and the FAQ answers AI assistants can quote.</p>\n<div class="card"><h3>Full suggested GBP description</h3><p>{esc(plan.get(\'gbp_description\'))}</p><p class="muted">{len(plan.get(\'gbp_description\',\'\'))}/750 characters</p></div>\n{\'\'.join(items)}\n\n<h2>3. Schema markup (JSON-LD)</h2>\n<div class="card">{prob_html}<p class="muted">Paste each block into Google\'s Rich Results Test (search.google.com/test/rich-results) and the Schema.org validator (validator.schema.org) before publishing. Both are already embedded in the generated landing page\'s &lt;head&gt;.</p></div>\n<h3>Restaurant</h3><pre>{esc(json.dumps(rest_ld, indent=2))}</pre>\n<h3>FAQPage (5 FAQs)</h3><dl class="faq card">{faq_all}</dl><pre>{esc(json.dumps(faq_ld, indent=2))}</pre>\n\n<h2>4. Manual rank check</h2>\n<div class="card"><p>Fill in <code>rank_tracker.csv</code> <b>before</b> making changes, then re-check every 2 weeks.</p><ol>\n<li>Open a private/incognito window, set the location to {esc(city_of(r.get(\'address\',\'\')))}, and search each keyword in Google Maps. Record the position (1-20 or "not found") and whether it is in the top 3 map pack.</li>\n<li>Ask ChatGPT, Gemini and Perplexity the keyword as a question (e.g. "Where can I get {esc(plan[\'top10\'][0][\'keyword\'])}?"). Record Y/N if {esc(r[\'name\'])} is mentioned and what the answer said.</li>\n<li>Log the change you made (GBP edit, page published, schema added) and the date.</li>\n<li>Never ask anyone to click, search repeatedly, or leave reviews in exchange for anything. Measure, don\'t manipulate.</li></ol>\n<table><tr><th>#</th><th>Keyword</th><th>Maps position (before)</th><th>Top 3?</th><th>AI mentions it?</th></tr>\n{\'\'.join(f"<tr><td>{t[\'rank\']}</td><td>{esc(t[\'keyword\'])}</td><td>__</td><td>☐</td><td>☐</td></tr>" for t in plan[\'top10\'])}</table></div>\n\n<h2>5. Honest methods</h2>\n<div class="honest">This plan only uses what Google and AI assistants reward on their own: accurate business info, real menu items, real photos, helpful pages, structured data, and asking <b>real</b> customers for honest reviews. No fake reviews, no fake clicks, no keyword stuffing (every page is linted for density &gt;3%), no incentivized reviews, and no requests to anyone to change rankings.</div>\n\n<h2>Files</h2><div class="card"><ul><li><code>landing.html</code> landing page with both schema blocks in &lt;head&gt;</li><li><code>plan.json</code> full plan</li><li><code>keywords_20.csv</code> all keywords</li><li><code>rank_tracker.csv</code> before/after tracker</li><li><code>schema_restaurant.jsonld</code>, <code>schema_faq.jsonld</code></li></ul></div>\n</div></body></html>"""\n\n\ndef render_landing(r: dict, plan: dict, rest_ld: dict, faq_ld: dict) -> str:\n    t = plan["top10"][0]\n    w = t["website"]\n    menu = "".join(f"<li><b>{esc(i.title())}</b> <span>[price]</span></li>" for i in split_list(r.get("menu_items", "")))\n    feats = "".join(f"<span class=\'chip\'>{esc(f)}</span>" for f in split_list(r.get("features", "")))\n    faq = "".join(f"<details><summary>{esc(f[\'q\'])}</summary><p>{esc(f[\'a\'])}</p></details>" for f in plan["faqs"][:5])\n    return f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\n<title>{esc(w[\'title_tag\'])}</title>\n<meta name="description" content="{esc(w[\'meta_description\'])}">\n<link rel="canonical" href="{esc(r.get(\'website\'))}/{slugify(t[\'keyword\'])}">\n<script type="application/ld+json">\n{json.dumps(rest_ld, indent=2)}\n</script>\n<script type="application/ld+json">\n{json.dumps(faq_ld, indent=2)}\n</script>\n<style>\nbody{{margin:0;font:17px/1.6 Georgia,serif;color:#2b2118;background:#fffaf3}}\n.hero{{background:linear-gradient(135deg,#7c2d12,#b45309);color:#fff;padding:70px 20px;text-align:center}}\n.hero h1{{font-size:40px;margin:0 0 10px}}.hero p{{font-family:sans-serif;opacity:.9}}\n.btn{{display:inline-block;background:#fff;color:#7c2d12;padding:12px 24px;border-radius:999px;text-decoration:none;font:600 16px sans-serif;margin:8px}}\nmain{{max-width:820px;margin:0 auto;padding:30px 20px 70px}}.chip{{display:inline-block;background:#fde68a;border-radius:999px;padding:4px 12px;margin:4px;font:14px sans-serif}}\n.menu li{{list-style:none;border-bottom:1px dashed #d6c4a8;padding:8px 0;display:flex;justify-content:space-between}}.menu{{padding:0}}\ndetails{{border-bottom:1px solid #eadcc8;padding:10px 0}}summary{{cursor:pointer;font-weight:bold}}\n.info{{background:#fff;border:1px solid #eadcc8;border-radius:12px;padding:16px;font-family:sans-serif}}\n</style></head><body>\n<section class="hero"><h1>{esc(w[\'h1\'])}</h1><p>{esc(r.get(\'cuisine\'))} · {esc(r.get(\'address\'))} · {esc(r.get(\'hours\'))}</p>\n<a class="btn" href="{esc(r.get(\'menu_link\'))}">See the menu</a><a class="btn" href="tel:{esc(r.get(\'phone\'))}">Call to order</a></section>\n<main><div style="text-align:center">{feats}</div>\n<article>{paragraphs(w[\'page_copy\'])}</article>\n<h2>Signature dishes</h2><ul class="menu">{menu}</ul>\n<div class="info"><b>{esc(r[\'name\'])}</b><br>{esc(r.get(\'address\'))}<br>Hours: {esc(r.get(\'hours\'))}<br>Menu: <a href="{esc(r.get(\'menu_link\'))}">{esc(r.get(\'menu_link\'))}</a></div>\n<h2>Frequently asked questions</h2>{faq}\n</main></body></html>"""\n\n\ndef check_plan_shape(plan: dict):\n    assert isinstance(plan.get("keywords"), list) and len(plan["keywords"]) >= 10, "need >=10 keywords"\n    assert isinstance(plan.get("top10"), list) and len(plan["top10"]) >= 1, "need top10"\n    plan["top10"] = plan["top10"][:10]\n    for i, t in enumerate(plan["top10"], 1):\n        t["rank"] = i\n        t.setdefault("gbp", {}); t.setdefault("website", {}); t.setdefault("aeo_faqs", [])\n        t.setdefault("intent_group", "dish")\n    for k in plan["keywords"]:\n        k.setdefault("intent_group", "dish"); k.setdefault("est_intent", "medium"); k.setdefault("intent_score", 5)\n    if len(plan.get("faqs", [])) < 5:\n        extra = [f for t in plan["top10"] for f in t["aeo_faqs"]]\n        plan["faqs"] = (plan.get("faqs", []) + extra)[:5]\n    plan.setdefault("gbp_description", "")\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description="Page One Builder: 10 keywords on Google Maps (SEO + AEO)")\n    ap.add_argument("--restaurant", help="restaurant name (built-in sample, or looked up in --csv)")\n    ap.add_argument("--csv", help="restaurants.csv (name,address,cuisine,website,menu_link[,phone,hours,features,menu_items])")\n    ap.add_argument("--row", type=int, help="0-indexed CSV row")\n    ap.add_argument("--offline", action="store_true", help="use cached sample output (no API key / wifi needed)")\n    ap.add_argument("--model", default=os.environ.get("GEMINI_MODEL", "gemini-2.5-flash"))\n    ap.add_argument("--out", default=str(ROOT / "out"))\n    ap.add_argument("--all", action="store_true", help="batch: run every row of --csv and write out/index.html")\n    a = ap.parse_args(argv)\n\n    if a.all:\n        if not a.csv:\n            sys.exit("--all needs --csv")\n        with open(a.csv, newline="", encoding="utf-8-sig") as f:\n            n = sum(1 for _ in csv.DictReader(f))\n        links = []\n        for i in range(n):\n            sub = ["--csv", a.csv, "--row", str(i), "--model", a.model, "--out", a.out] + (["--offline"] if a.offline else [])\n            d = main(sub)\n            links.append(f"<li><a href=\'{d.name}/report.html\'>{esc(d.name)}</a> · <a href=\'{d.name}/landing.html\'>landing</a></li>")\n        idx = Path(a.out) / "index.html"\n        idx.write_text(f"<!doctype html><meta charset=utf-8><title>Page One Builder</title><style>{CSS}</style><div class=wrap><h1>Page One plans</h1><ul>{\'\'.join(links)}</ul></div>")\n        print(f"→ Batch index: {idx}")\n        return Path(a.out)\n\n    if a.csv:\n        r = load_csv_row(a.csv, a.row, a.restaurant)\n    elif a.restaurant and a.restaurant.lower() != SAMPLE_RESTAURANT["name"].lower():\n        r = {k: "" for k in SAMPLE_RESTAURANT}; r["name"] = a.restaurant\n        print("! Only a name was given; pass --csv for address/cuisine/menu for better results.")\n    else:\n        r = dict(SAMPLE_RESTAURANT)\n\n    meta = {"date": date.today().isoformat()}\n    plan = None\n    if not a.offline:\n        if not get_api_key():\n            print("! GEMINI_API_KEY not found (env, .env, or Colab secrets). Falling back to --offline sample output.")\n        else:\n            try:\n                print(f"→ Asking {a.model} for 20 keywords + 10-keyword plan for {r[\'name\']} ...")\n                plan = call_gemini(r, a.model)\n                check_plan_shape(plan)\n                meta.update(mode="live", engine=a.model)\n            except Exception as e:\n                print(f"! Gemini call failed ({type(e).__name__}: {str(e)[:200]}). Falling back to offline sample output.")\n                plan = None\n    if plan is None:\n        plan = json.loads(SAMPLE_JSON.read_text())\n        meta.update(mode="offline", engine="cached sample", source_note=plan.get("_note", ""))\n        if r["name"] != SAMPLE_RESTAURANT["name"]:\n            print(f"! Offline mode only has cached output for {SAMPLE_RESTAURANT[\'name\']}; using that restaurant.")\n            r = dict(SAMPLE_RESTAURANT)\n        check_plan_shape(plan)\n\n    rest_ld = build_restaurant_schema(r, plan)\n    faq_ld = build_faq_schema(plan["faqs"])\n    problems = validate_schema(rest_ld, faq_ld)\n    lint = lint_plan(plan)\n\n    outdir = Path(a.out) / slugify(r["name"])\n    outdir.mkdir(parents=True, exist_ok=True)\n    (outdir / "plan.json").write_text(json.dumps({"restaurant": r, "meta": meta, **plan, "lint": lint, "schema_problems": problems}, indent=2))\n    (outdir / "schema_restaurant.jsonld").write_text(json.dumps(rest_ld, indent=2))\n    (outdir / "schema_faq.jsonld").write_text(json.dumps(faq_ld, indent=2))\n    write_csvs(outdir, plan, r)\n    (outdir / "report.html").write_text(render_report(r, plan, rest_ld, faq_ld, problems, lint, meta))\n    (outdir / "landing.html").write_text(render_landing(r, plan, rest_ld, faq_ld))\n\n    # final validation: the schema embedded in the landing page must parse\n    lp = (outdir / "landing.html").read_text()\n    blocks = re.findall(r\'<script type="application/ld\\+json">(.*?)</script>\', lp, re.S)\n    parsed = [json.loads(b) for b in blocks]\n    print(f"✓ Mode: {meta[\'mode\']} ({meta[\'engine\']})")\n    print(f"✓ {len(plan[\'keywords\'])} keywords, top {len(plan[\'top10\'])} planned")\n    print(f"✓ JSON-LD in landing page: {len(parsed)} blocks parsed OK ({\', \'.join(p[\'@type\'] for p in parsed)})")\n    for p in problems:\n        print(f"  ⚠ {p}")\n    flagged = [l for l in lint if l["flags"]]\n    print(f"✓ Honesty lint: {10 - len(flagged)}/{len(lint)} pages clean" + ("" if not flagged else f" ({len(flagged)} flagged, see report)"))\n    print(f"→ Report:  {outdir / \'report.html\'}")\n    print(f"→ Landing: {outdir / \'landing.html\'}")\n    print(f"→ Data:    plan.json, keywords_20.csv, rank_tracker.csv, schema_*.jsonld in {outdir}")\n    return outdir\n\n\nif __name__ == "__main__":\n    main()\n')
pathlib.Path('sample_output/sample_chettinad_kitchen.plan.json').write_text('{\n  "_note": "Hand-written sample output for demo/offline mode (no Gemini call). Prices shown as [price] placeholders; phone and street address must be filled from restaurants.csv before publishing.",\n  "keywords": [\n    {\n      "keyword": "chicken biryani near me",\n      "intent_group": "dish",\n      "est_intent": "high",\n      "intent_score": 10,\n      "why": "Top dish search; people ready to order now"\n    },\n    {\n      "keyword": "chettinad chicken richardson",\n      "intent_group": "dish",\n      "est_intent": "high",\n      "intent_score": 9,\n      "why": "Signature dish + city = strong local intent"\n    },\n    {\n      "keyword": "dosa near me",\n      "intent_group": "dish",\n      "est_intent": "high",\n      "intent_score": 9,\n      "why": "High-volume South Indian dish search"\n    },\n    {\n      "keyword": "best biryani in richardson tx",\n      "intent_group": "dish",\n      "est_intent": "high",\n      "intent_score": 9,\n      "why": "\'Best\' + dish + city is a classic Maps query"\n    },\n    {\n      "keyword": "masala dosa richardson",\n      "intent_group": "dish",\n      "est_intent": "medium",\n      "intent_score": 7,\n      "why": "Specific dosa variant searchers"\n    },\n    {\n      "keyword": "spicy south indian chicken curry",\n      "intent_group": "dish",\n      "est_intent": "medium",\n      "intent_score": 6,\n      "why": "Describes Chettinad food without knowing the name"\n    },\n    {\n      "keyword": "goat biryani dallas",\n      "intent_group": "dish",\n      "est_intent": "medium",\n      "intent_score": 6,\n      "why": "Broader metro dish search"\n    },\n    {\n      "keyword": "south indian restaurant near me",\n      "intent_group": "cuisine_near_me",\n      "est_intent": "high",\n      "intent_score": 10,\n      "why": "Core cuisine + near me"\n    },\n    {\n      "keyword": "chettinad restaurant near me",\n      "intent_group": "cuisine_near_me",\n      "est_intent": "high",\n      "intent_score": 8,\n      "why": "Exact niche match, low competition"\n    },\n    {\n      "keyword": "indian restaurant richardson tx",\n      "intent_group": "cuisine_near_me",\n      "est_intent": "high",\n      "intent_score": 9,\n      "why": "Broad cuisine + city"\n    },\n    {\n      "keyword": "tamil food near me",\n      "intent_group": "cuisine_near_me",\n      "est_intent": "medium",\n      "intent_score": 7,\n      "why": "Chettinad is Tamil cuisine; diaspora search"\n    },\n    {\n      "keyword": "indian food open now",\n      "intent_group": "cuisine_near_me",\n      "est_intent": "high",\n      "intent_score": 8,\n      "why": "Time-sensitive intent"\n    },\n    {\n      "keyword": "halal indian restaurant richardson",\n      "intent_group": "occasion_feature",\n      "est_intent": "high",\n      "intent_score": 9,\n      "why": "Halal is a must-have filter for many diners"\n    },\n    {\n      "keyword": "indian lunch buffet near me",\n      "intent_group": "occasion_feature",\n      "est_intent": "high",\n      "intent_score": 9,\n      "why": "Lunch buffet is a strong weekday driver"\n    },\n    {\n      "keyword": "late night food near ut dallas",\n      "intent_group": "occasion_feature",\n      "est_intent": "high",\n      "intent_score": 8,\n      "why": "Open till 11 PM + student audience"\n    },\n    {\n      "keyword": "cheap eats near ut dallas",\n      "intent_group": "occasion_feature",\n      "est_intent": "medium",\n      "intent_score": 7,\n      "why": "Students on a budget"\n    },\n    {\n      "keyword": "indian food open late richardson",\n      "intent_group": "occasion_feature",\n      "est_intent": "high",\n      "intent_score": 8,\n      "why": "Late-hours feature + city"\n    },\n    {\n      "keyword": "halal biryani near me",\n      "intent_group": "occasion_feature",\n      "est_intent": "high",\n      "intent_score": 9,\n      "why": "Combines feature + top dish"\n    },\n    {\n      "keyword": "group dinner indian restaurant richardson",\n      "intent_group": "occasion_feature",\n      "est_intent": "medium",\n      "intent_score": 6,\n      "why": "Groups / student orgs"\n    },\n    {\n      "keyword": "vegetarian south indian food richardson",\n      "intent_group": "occasion_feature",\n      "est_intent": "medium",\n      "intent_score": 6,\n      "why": "Dosa makes it veg-friendly"\n    }\n  ],\n  "top10": [\n    {\n      "rank": 1,\n      "keyword": "chicken biryani near me",\n      "intent_group": "dish",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Indian restaurant",\n          "Halal restaurant"\n        ],\n        "description_addition": "Signature dish highlighted in the description in plain language, e.g. \'Known for Chettinad-style chicken biryani and pepper chicken.\'",\n        "menu_items": [\n          "Chicken Biryani",\n          "Chettinad Chicken",\n          "Masala Dosa",\n          "Goat Biryani"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'chicken biryani near me\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about chicken biryani with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Chicken Biryani in Richardson, Made the Chettinad Way",\n        "meta_description": "Looking for chicken biryani near you in Richardson? Our biryani is cooked in the Chettinad style, with whole spices roasted in-house and seeraga...",\n        "h1": "Chicken Biryani in Richardson, Made the Chettinad Way",\n        "page_copy": "Looking for chicken biryani near you in Richardson? Our biryani is cooked in the Chettinad style, with whole spices roasted in-house and seeraga samba or basmati rice layered over tender halal chicken. It is the dish most guests order on their first visit.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "Does Sample Chettinad Kitchen serve chicken biryani?",\n          "a": "Yes. Chicken biryani is a signature dish, made Chettinad-style with halal chicken and served with raita and salna."\n        },\n        {\n          "q": "Is the chicken biryani halal?",\n          "a": "Yes, all meat at Sample Chettinad Kitchen is halal."\n        },\n        {\n          "q": "Can I order biryani for pickup in Richardson?",\n          "a": "Yes. Call ahead or order at the counter for pickup in Richardson, TX 75080."\n        }\n      ]\n    },\n    {\n      "rank": 2,\n      "keyword": "south indian restaurant near me",\n      "intent_group": "cuisine_near_me",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Indian restaurant",\n          "Tamil restaurant"\n        ],\n        "description_addition": "State the cuisine and location clearly: \'Chettinad (South Indian) restaurant in Richardson, near UT Dallas.\'",\n        "menu_items": [\n          "Chettinad Chicken",\n          "Chicken Biryani",\n          "Dosa",\n          "Idli"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'south indian restaurant near me\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about south indian restaurant with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "South Indian Food in Richardson, TX",\n        "meta_description": "If you are searching for a South Indian restaurant close to Richardson or UT Dallas, we cook the food of Tamil Nadu\'s Chettinad region: peppery...",\n        "h1": "South Indian Food in Richardson, TX",\n        "page_copy": "If you are searching for a South Indian restaurant close to Richardson or UT Dallas, we cook the food of Tamil Nadu\'s Chettinad region: peppery curries, biryani, dosa and idli, made fresh every day.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "What kind of food does Sample Chettinad Kitchen serve?",\n          "a": "South Indian food from the Chettinad region of Tamil Nadu: biryani, Chettinad chicken, dosa and more."\n        },\n        {\n          "q": "Where is Sample Chettinad Kitchen?",\n          "a": "Richardson, TX 75080, a short drive from UT Dallas."\n        },\n        {\n          "q": "Does it have vegetarian options?",\n          "a": "Yes. Dosa and several curries are vegetarian."\n        }\n      ]\n    },\n    {\n      "rank": 3,\n      "keyword": "halal indian restaurant richardson",\n      "intent_group": "occasion_feature",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Halal restaurant",\n          "Buffet restaurant"\n        ],\n        "description_addition": "Mention the real feature once: halal meat, weekday lunch buffet, open until 11 PM.",\n        "menu_items": [\n          "Lunch Buffet (weekdays)",\n          "Halal Chicken Biryani",\n          "Late-night Dosa"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'halal indian restaurant richardson\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about halal indian restaurant richardson with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Halal Indian Restaurant in Richardson",\n        "meta_description": "Every meat dish on our menu is made with halal chicken and goat, so you can order the full menu with confidence. We are a family-style South Indian...",\n        "h1": "Halal Indian Restaurant in Richardson",\n        "page_copy": "Every meat dish on our menu is made with halal chicken and goat, so you can order the full menu with confidence. We are a family-style South Indian restaurant in Richardson serving Chettinad cooking.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "Is Sample Chettinad Kitchen halal?",\n          "a": "Yes, all chicken and goat served are halal."\n        },\n        {\n          "q": "Is the lunch buffet halal?",\n          "a": "Yes, every meat dish on the buffet is halal."\n        },\n        {\n          "q": "What halal dishes are popular?",\n          "a": "Chicken biryani and Chettinad chicken are the most ordered halal dishes."\n        }\n      ]\n    },\n    {\n      "rank": 4,\n      "keyword": "indian lunch buffet near me",\n      "intent_group": "occasion_feature",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Halal restaurant",\n          "Buffet restaurant"\n        ],\n        "description_addition": "Mention the real feature once: halal meat, weekday lunch buffet, open until 11 PM.",\n        "menu_items": [\n          "Lunch Buffet (weekdays)",\n          "Halal Chicken Biryani",\n          "Late-night Dosa"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'indian lunch buffet near me\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about indian lunch buffet with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Weekday South Indian Lunch Buffet in Richardson",\n        "meta_description": "Our weekday lunch buffet is built for people with an hour to eat: biryani, two or three Chettinad curries, a vegetarian dish, dosa made to order at...",\n        "h1": "Weekday South Indian Lunch Buffet in Richardson",\n        "page_copy": "Our weekday lunch buffet is built for people with an hour to eat: biryani, two or three Chettinad curries, a vegetarian dish, dosa made to order at the counter, and dessert, at one set price [price].\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "Does Sample Chettinad Kitchen have a lunch buffet?",\n          "a": "Yes, a South Indian lunch buffet runs on weekdays."\n        },\n        {\n          "q": "What is on the lunch buffet?",\n          "a": "Biryani, rotating Chettinad curries, a vegetarian dish, made-to-order dosa and dessert."\n        },\n        {\n          "q": "Is the buffet good for a quick lunch near UT Dallas?",\n          "a": "Yes. It is a few minutes from campus and designed for a one-hour lunch."\n        }\n      ]\n    },\n    {\n      "rank": 5,\n      "keyword": "chettinad chicken richardson",\n      "intent_group": "dish",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Indian restaurant",\n          "Halal restaurant"\n        ],\n        "description_addition": "Signature dish highlighted in the description in plain language, e.g. \'Known for Chettinad-style chicken biryani and pepper chicken.\'",\n        "menu_items": [\n          "Chicken Biryani",\n          "Chettinad Chicken",\n          "Masala Dosa",\n          "Goat Biryani"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'chettinad chicken richardson\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about chettinad chicken richardson with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Chettinad Chicken in Richardson, TX",\n        "meta_description": "Chettinad chicken is the signature dish of our kitchen and the reason we opened in Richardson. It is a dark, pepper-forward curry made with a masala...",\n        "h1": "Chettinad Chicken in Richardson, TX",\n        "page_copy": "Chettinad chicken is the signature dish of our kitchen and the reason we opened in Richardson. It is a dark, pepper-forward curry made with a masala of more than ten spices that we roast and grind ourselves.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "What is Chettinad chicken?",\n          "a": "A pepper-forward South Indian chicken curry made with freshly roasted Chettinad spices."\n        },\n        {\n          "q": "Where can I get Chettinad chicken in Richardson?",\n          "a": "Sample Chettinad Kitchen in Richardson, TX 75080 serves it daily."\n        },\n        {\n          "q": "Is Chettinad chicken very spicy?",\n          "a": "It is peppery; the kitchen can make it milder on request."\n        }\n      ]\n    },\n    {\n      "rank": 6,\n      "keyword": "dosa near me",\n      "intent_group": "dish",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Indian restaurant",\n          "Halal restaurant"\n        ],\n        "description_addition": "Signature dish highlighted in the description in plain language, e.g. \'Known for Chettinad-style chicken biryani and pepper chicken.\'",\n        "menu_items": [\n          "Chicken Biryani",\n          "Chettinad Chicken",\n          "Masala Dosa",\n          "Goat Biryani"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'dosa near me\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about dosa with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Crispy Dosa in Richardson, Near UT Dallas",\n        "meta_description": "Craving a hot, crispy dosa nearby? We make every dosa to order on a cast-iron tawa from batter fermented overnight, and serve it with sambar and...",\n        "h1": "Crispy Dosa in Richardson, Near UT Dallas",\n        "page_copy": "Craving a hot, crispy dosa nearby? We make every dosa to order on a cast-iron tawa from batter fermented overnight, and serve it with sambar and coconut, tomato and mint chutneys.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "Does Sample Chettinad Kitchen make dosa?",\n          "a": "Yes, dosa is made to order and served with sambar and chutneys."\n        },\n        {\n          "q": "What dosa types are available?",\n          "a": "Plain, masala, onion and ghee roast dosa."\n        },\n        {\n          "q": "Is dosa vegetarian?",\n          "a": "Yes, dosa is vegetarian and made from fermented rice and lentil batter."\n        }\n      ]\n    },\n    {\n      "rank": 7,\n      "keyword": "late night food near ut dallas",\n      "intent_group": "occasion_feature",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Halal restaurant",\n          "Buffet restaurant"\n        ],\n        "description_addition": "Mention the real feature once: halal meat, weekday lunch buffet, open until 11 PM.",\n        "menu_items": [\n          "Lunch Buffet (weekdays)",\n          "Halal Chicken Biryani",\n          "Late-night Dosa"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'late night food near ut dallas\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about late night food with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Late-Night South Indian Food Near UT Dallas",\n        "meta_description": "Studying late or just finished practice? We are open until 11 PM every day, a few minutes from UT Dallas, with hot biryani, dosa and Chettinad...",\n        "h1": "Late-Night South Indian Food Near UT Dallas",\n        "page_copy": "Studying late or just finished practice? We are open until 11 PM every day, a few minutes from UT Dallas, with hot biryani, dosa and Chettinad curries for dine-in or pickup.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "How late is Sample Chettinad Kitchen open?",\n          "a": "Open until 11 PM daily."\n        },\n        {\n          "q": "Is it close to UT Dallas?",\n          "a": "Yes, it is in Richardson, a short drive from campus."\n        },\n        {\n          "q": "Can I get takeout late at night?",\n          "a": "Yes, pickup is available until closing."\n        }\n      ]\n    },\n    {\n      "rank": 8,\n      "keyword": "indian restaurant richardson tx",\n      "intent_group": "cuisine_near_me",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Indian restaurant",\n          "Tamil restaurant"\n        ],\n        "description_addition": "State the cuisine and location clearly: \'Chettinad (South Indian) restaurant in Richardson, near UT Dallas.\'",\n        "menu_items": [\n          "Chettinad Chicken",\n          "Chicken Biryani",\n          "Dosa",\n          "Idli"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'indian restaurant richardson tx\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about indian restaurant richardson tx with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Indian Restaurant in Richardson, TX",\n        "meta_description": "Among Indian restaurants in Richardson, TX, we focus on one region\'s cooking, Chettinad from South India, instead of a generic menu. That focus shows...",\n        "h1": "Indian Restaurant in Richardson, TX: Chettinad Cuisine",\n        "page_copy": "Among Indian restaurants in Richardson, TX, we focus on one region\'s cooking, Chettinad from South India, instead of a generic menu. That focus shows up in the spice blends, the biryani and the dosa.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "What Indian cuisine does Sample Chettinad Kitchen specialize in?",\n          "a": "Chettinad cuisine from South India."\n        },\n        {\n          "q": "What are the most popular dishes?",\n          "a": "Chicken biryani, Chettinad chicken and dosa."\n        },\n        {\n          "q": "Does it take reservations?",\n          "a": "Yes, reservations are accepted, including for groups."\n        }\n      ]\n    },\n    {\n      "rank": 9,\n      "keyword": "halal biryani near me",\n      "intent_group": "occasion_feature",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Halal restaurant",\n          "Buffet restaurant"\n        ],\n        "description_addition": "Mention the real feature once: halal meat, weekday lunch buffet, open until 11 PM.",\n        "menu_items": [\n          "Lunch Buffet (weekdays)",\n          "Halal Chicken Biryani",\n          "Late-night Dosa"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'halal biryani near me\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about halal biryani with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "Halal Biryani in Richardson | Sample Chettinad Kitchen",\n        "meta_description": "Our biryani is made only with halal chicken and goat, cooked dum-style so the rice absorbs the masala. Guests who want halal biryani nearby in...",\n        "h1": "Halal Biryani in Richardson",\n        "page_copy": "Our biryani is made only with halal chicken and goat, cooked dum-style so the rice absorbs the masala. Guests who want halal biryani nearby in Richardson and around UT Dallas order it for dine-in, pickup and parties.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "Is the biryani at Sample Chettinad Kitchen halal?",\n          "a": "Yes, chicken and goat biryani are made with halal meat."\n        },\n        {\n          "q": "Can I order biryani for a party?",\n          "a": "Yes, call ahead for large or party orders."\n        },\n        {\n          "q": "What comes with the biryani?",\n          "a": "Raita and salna."\n        }\n      ]\n    },\n    {\n      "rank": 10,\n      "keyword": "chettinad restaurant near me",\n      "intent_group": "cuisine_near_me",\n      "gbp": {\n        "categories": [\n          "South Indian restaurant",\n          "Indian restaurant",\n          "Tamil restaurant"\n        ],\n        "description_addition": "State the cuisine and location clearly: \'Chettinad (South Indian) restaurant in Richardson, near UT Dallas.\'",\n        "menu_items": [\n          "Chettinad Chicken",\n          "Chicken Biryani",\n          "Dosa",\n          "Idli"\n        ],\n        "photos": [\n          "Close-up of the dish/experience for \'chettinad restaurant near me\' in natural light",\n          "Interior with guests (with permission)",\n          "Exterior + signage so people recognize the storefront",\n          "Menu board showing prices"\n        ],\n        "posts": [\n          "Google Post: photo + 2 lines about chettinad restaurant with a \'Call\' button",\n          "Weekly \'today\'s special\' post mentioning hours (open till 11 PM)"\n        ],\n        "review_request": "Add a QR code on receipts/tables: \'Enjoyed your meal? An honest Google review helps us a lot.\' Ask every customer the same way; no discounts or gifts for reviews, and never ask for a specific rating or wording."\n      },\n      "website": {\n        "title_tag": "A Chettinad Restaurant in Richardson",\n        "meta_description": "Chettinad restaurants are rare in North Texas. We bring the cooking of Karaikudi and the Chettinad region to Richardson: roasted spice masalas,...",\n        "h1": "A Chettinad Restaurant in Richardson",\n        "page_copy": "Chettinad restaurants are rare in North Texas. We bring the cooking of Karaikudi and the Chettinad region to Richardson: roasted spice masalas, pepper chicken, kuzhambu, biryani and dosa.\\n\\nOur kitchen follows the cooking of the Chettinad region of Tamil Nadu, known for freshly roasted and ground spices: black pepper, fennel, star anise, dried red chilies, kalpasi (stone flower) and curry leaves. We roast our masalas in small batches every morning, so the flavor is deep and aromatic rather than just hot. If you prefer milder food, tell your server and the kitchen will adjust the spice level for you.\\n\\nThe menu centers on three dishes our regulars come back for. Chicken biryani [price] is layered with seeded basmati rice, slow-cooked chicken, mint and fried onions, and it comes with raita and salna. Chettinad chicken [price] is a dry-roasted pepper curry that pairs well with parotta or steamed rice. Dosa [price] is a thin, crisp rice-and-lentil crepe served with sambar and fresh chutneys, with masala, ghee roast and onion versions available.\\n\\nAll of our meat is halal. On weekdays we run a lunch buffet with rotating curries, biryani, dosa made to order and dessert, a quick option if you are coming from class or the office. We stay open until 11 PM, so dinner after a late lab or a game is easy.\\n\\nYou will find us in Richardson, TX 75080, a short drive from the UT Dallas campus. Dine in, pick up, or call ahead for large group orders. Hours: open daily from 11 AM to 11 PM. See the full menu with current prices on our menu page, and check our Google Business Profile for photos and today\'s specials.\\n\\nQuestions before you visit? Read the FAQ below or call us. We would love to cook for you."\n      },\n      "aeo_faqs": [\n        {\n          "q": "Is there a Chettinad restaurant in Richardson?",\n          "a": "Yes, Sample Chettinad Kitchen in Richardson, TX 75080."\n        },\n        {\n          "q": "What makes Chettinad food different?",\n          "a": "Freshly roasted and ground spices like black pepper, fennel and kalpasi."\n        },\n        {\n          "q": "What should I order first?",\n          "a": "Chettinad chicken with parotta, or the chicken biryani."\n        }\n      ]\n    }\n  ],\n  "gbp_description": "Sample Chettinad Kitchen is a South Indian restaurant in Richardson, TX, serving the cooking of Tamil Nadu\'s Chettinad region. Known for chicken biryani, Chettinad pepper chicken and crispy dosa made to order. All meat is halal. Weekday lunch buffet, and we\'re open daily until 11 PM, a short drive from UT Dallas. Dine in, pickup, and group orders welcome.",\n  "faqs": [\n    {\n      "q": "Is Sample Chettinad Kitchen halal?",\n      "a": "Yes. All chicken and goat served at Sample Chettinad Kitchen are halal."\n    },\n    {\n      "q": "What are the hours of Sample Chettinad Kitchen?",\n      "a": "Open daily from 11 AM to 11 PM."\n    },\n    {\n      "q": "Does Sample Chettinad Kitchen have a lunch buffet?",\n      "a": "Yes, a South Indian lunch buffet is served on weekdays."\n    },\n    {\n      "q": "Where is Sample Chettinad Kitchen located?",\n      "a": "Richardson, TX 75080, a short drive from UT Dallas."\n    },\n    {\n      "q": "What dishes is Sample Chettinad Kitchen known for?",\n      "a": "Chicken biryani, Chettinad chicken and dosa."\n    }\n  ]\n}')
print('ready')

## Choose a restaurant
Upload the organizers' `restaurants.csv` (Files panel) and set `CSV` + `ROW`, or leave `CSV = None` to use Sample Chettinad Kitchen.

In [ ]:
CSV = None          # e.g. 'restaurants.csv'
ROW = 0
OFFLINE = False     # True = cached sample, no API call

import sys, os; sys.path.insert(0, os.getcwd())
import page_one, importlib; importlib.reload(page_one)
args = []
if CSV: args += ['--csv', CSV, '--row', str(ROW)]
else: args += ['--restaurant', 'Sample Chettinad Kitchen']
if OFFLINE: args.append('--offline')
outdir = page_one.main(args)

In [ ]:
from IPython.display import HTML, display
display(HTML(open(outdir/'report.html').read()))

In [ ]:
import json
print(open(outdir/'schema_restaurant.jsonld').read())
print(open(outdir/'schema_faq.jsonld').read())

In [ ]:
import shutil
shutil.make_archive('page_one_output','zip',outdir)
try:
    from google.colab import files; files.download('page_one_output.zip')
except Exception: print('Saved page_one_output.zip')